# 01 - Build Raw PBMC Object
### By [Mansi Singh](mansi.singh@alleninstitute.org), Comp Bio, Allen Institute for Immunology
### Original notebook by [Aishwarya Chander](aishwarya.chander@alleninstitute.org), High Resolution Translational Immunology, Allen Institute for Immunology

**Main aim**: 
In this notebook, I take my previously cleaned up metadata, predicted doublets and CellTypist labels and add them onto an object that contains all non-DARA PBMC samples. This will act as my base object for further PBMC analyses. I also save all objects per sample.

In [1]:
import os
import warnings
from concurrent.futures import ThreadPoolExecutor, as_completed
from functools import reduce

import anndata
import h5py
import hisepy as hp
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import scipy.sparse as scs
from tqdm import tqdm

# Settings
warnings.filterwarnings("ignore")  # Suppress deprecation warnings for cleaner output
sc.settings.n_jobs = 60  # Parallelism tuned for HISE compute environment (60 cores)
plt.style.use("default")

## 1. Setup

### 1.0 Necessary Paths

In [2]:
base_path = "../../../data/rna/"

celltypist_labels_dir = base_path + "celltypist/labels/"
file_path_csvs = base_path + "metadata/"

### 1.1 Custom Functions

In [3]:
def read_mat(h5_con, mat_name):
    """Reads a sparse expression matrix from an HDF5 file."""
    mat = scs.csc_matrix(
        (
            h5_con[mat_name]["data"][:],
            h5_con[mat_name]["indices"][:],
            h5_con[mat_name]["indptr"][:],
        ),
        shape=tuple(h5_con[mat_name]["shape"][:]),
    )
    return mat


def read_feats(h5_con, mat_name, name_col):
    """Reads gene feature names from an HDF5 file."""
    feats = h5_con[mat_name]["features"][name_col][:]
    feats = [x.decode("UTF-8") for x in feats]
    return feats


def read_obs(h5con):
    """Reads per-cell observation metadata from an HDF5 file."""
    bc = h5con["matrix"]["barcodes"][:]
    bc = [x.decode("UTF-8") for x in bc]
    obs_df = pd.DataFrame({"barcodes": bc})
    obs_columns = h5con["matrix"]["observations"].keys()
    for col in obs_columns:
        values = h5con["matrix"]["observations"][col][:]
        if isinstance(values[0], (bytes, bytearray)):
            values = [x.decode("UTF-8") for x in values]
        obs_df[col] = values
    return obs_df

In [4]:
## CellTypist labels path — ensure this matches output from preprocessing notebooks
celltypist_labels_dir = base_path + "celltypist/labels/"


def get_labels_and_scores(
    pbmc_sample_id, data_sources, celltypist_labels_dir=celltypist_labels_dir, levels=3
):
    """
    For each pbmc_sample_id, extract labels predicted by different methods and combine them into a DataFrame.

    Parameters:
    pbmc_sample_id (str): The sample id to process.
    data_sources (list): List of data sources for different methods.
    levels (int): Number of levels of labels to process. Default is 3.

    Returns:
    DataFrame: DataFrame with the combined labels.
    """

    def load_and_merge(data_source):
        label_dfs = [
            pd.read_csv(
                f"{celltypist_labels_dir}{pbmc_sample_id}_l{i}_{data_source}_predicted_labels.csv"
            )[["barcodes", "predicted_labels"]]
            for i in range(1, levels + 1)
        ]

        merged_label_df = reduce(
            lambda left, right: pd.merge(left, right, on="barcodes", how="left"),
            label_dfs,
        )
        merged_label_df.columns = ["barcodes"] + [
            f"{data_source}_l{i}" for i in range(1, levels + 1)
        ]

        return merged_label_df

    merged_labels = [load_and_merge(data_source) for data_source in data_sources]

    return reduce(
        lambda left, right: pd.merge(left, right, on="barcodes", how="left"),
        merged_labels,
    )


def add_metadata(adata, metadata, doublet_scores_df):
    """
    Adds CellTypist labels, doublet scores, and clinical metadata to an AnnData object.

    Parameters:
    adata (AnnData): The AnnData object to annotate.
    metadata (DataFrame): Clinical/sample metadata with 'pbmc_sample_id' key.
    doublet_scores_df (DataFrame): Pre-computed doublet scores with 'barcodes' key.

    Returns:
    AnnData: The annotated AnnData object.
    """
    pbmc_sample_id = adata.obs["pbmc_sample_id"][0]
    df = get_labels_and_scores(pbmc_sample_id, ["aifi"])

    # Merge doublet scores onto the labels dataframe
    df = df.merge(doublet_scores_df, on="barcodes", how="left")
    adata.obs = adata.obs.merge(df, on="barcodes", how="left")

    # Clinical and sample metadata columns to attach
    add_meta_cols = [
        "pbmc_sample_id",
        "sample.sampleKitGuid",
        "sample.visitDetails",
        "sample.visitName",
        "sample.drawDate",
        "sample.daysSinceFirstVisit",
        "sample.diseaseStatesRecordedAtVisit",
        "subject.biologicalSex",
        "subject.birthYear",
        "subject.ethnicity",
        "subject.partnerCode",
        "subject.race",
        "subject.subjectGuid",
        "specimen.specimenGuid",
        "cohort.cohortGuid",
        "manual.time_stamp",
        "tissue",
        "manual.response",
        "manual.response_type",
        "manual.extracted_name",
        "manual.batch_id",
        "manual.category",
        "manual.treatment_dara",
        "manual.flu_response",
    ]

    add_meta = metadata[add_meta_cols]
    adata.obs = adata.obs.merge(add_meta, on="pbmc_sample_id", how="left")
    adata.obs = adata.obs.drop("barcodes", axis=1)
    return adata


## Raw h5 files directory — ensure this exists before running
pbmc_raw_dir = base_path + "raw-files/"
if not os.path.exists(pbmc_raw_dir):
    os.makedirs(pbmc_raw_dir)


def build_adata(
    h5_file, metadata, doublet_scores_df, pbmc_raw_dir=pbmc_raw_dir, save=False
):
    """
    Builds an AnnData object from an HDF5 file, attaching metadata and doublet scores.

    Parameters:
    h5_file (str): Path to the .h5 expression matrix file.
    metadata (DataFrame): Clinical/sample metadata.
    doublet_scores_df (DataFrame): Pre-computed doublet scores.
    pbmc_raw_dir (str): Output directory for per-sample h5ad files.
    save (bool): Whether to write the per-sample h5ad to disk.

    Returns:
    AnnData: Assembled AnnData object with expression data, metadata, and annotations.
    """
    h5_con = h5py.File(h5_file, mode="r")
    rna_mat = read_mat(h5_con, "matrix")
    obs = read_obs(h5_con)
    obs = obs.reset_index(drop=True)
    barcodes = obs["barcodes"]
    obs = obs.drop("barcodes", axis=1)
    genes = read_feats(h5_con, "matrix", "name")
    h5_con.close()
    adata = sc.AnnData(X=rna_mat.T, obs=obs)
    adata.var_names = genes
    adata.var_names_make_unique()
    adata.obs_names = barcodes

    adata = add_metadata(adata, metadata, doublet_scores_df)
    adata.obs_names = barcodes

    if save:
        pbmc_sample_id = adata.obs["pbmc_sample_id"][0]
        adata.write_h5ad(f"{pbmc_raw_dir}adata_raw_pbmc_{pbmc_sample_id}.h5ad")

    return adata

## 2. Load Metadata

In [5]:
metadata = pd.read_csv(base_path + "metadata/scrna_metadata.csv", index_col=0)
# Filter for PBMC tissue only; exclude dara-treated subjects
metadata = metadata[
    (metadata["tissue"] == "PBMC") & (metadata["manual.treatment_dara"] == "non_dara")
]

# Fill NaN with string 'None' for consistent downstream categorical handling
metadata = metadata.fillna("None")
file_paths = metadata["manual.file_paths"].to_list()

In [6]:
# Verify category distribution (expect: Treatment, Healthy)
metadata['manual.category'].value_counts()

healthy_pbmc    204
tumor_pbmc      166
Name: manual.category, dtype: int64

In [7]:
# Load pre-computed doublet scores for all samples (generated by scrublet)
doublet_scores_df = pd.read_parquet(base_path+'metadata/all_doublet_scores.parquet')

In [8]:
# Verify CellTypist label files are available (expect 3 files per sample × 193 samples)
def count_files_in_directory(directory_path):
    return len([name for name in os.listdir(directory_path) if os.path.isfile(os.path.join(directory_path, name))])

file_path_labels = base_path+'celltypist/labels/'
file_count = count_files_in_directory(file_path_labels)
print(f'There are {file_count} files in the directory.')

There are 13122 files in the directory.


In [9]:
files = list(metadata['file.id'])
len(files)

370

## 3. Build and Save Master Object

In [10]:
# Build AnnData objects in parallel; max_workers=4 to limit memory pressure from large h5 files
adata_list = []

with ThreadPoolExecutor(max_workers=4) as executor:
    future_to_file = {
        executor.submit(
            build_adata,
            file_path,
            metadata,
            doublet_scores_df,
            pbmc_raw_dir=pbmc_raw_dir,
            save=True,
        ): file_path
        for file_path in file_paths
    }
    for future in tqdm(as_completed(future_to_file), total=len(file_paths)):
        result = future.result()
        if result is not None:
            adata_list.append(result)

100%|██████████| 370/370 [52:26<00:00,  8.50s/it] 


In [12]:
# Concatenate all per-sample AnnData objects into a single master object
adata = anndata.concat(adata_list)

# Store sample-level metadata in .uns for downstream reference (e.g., clinical covariate lookups)
adata.uns["all_sample_metadata"] = metadata.set_index("pbmc_sample_id").to_dict(
    orient="index"
)

In [13]:
adata.write_h5ad(pbmc_raw_dir+'all-pbmc-raw.h5ad')
adata.obs.to_parquet(file_path_csvs+'all-pbmc-raw-metadata.parquet')